# Дополнительный семинар. Стек и очередь

Отменяя изменения в редакторе, мы начинаем с последнего. Обслуживая заявки, начинаем с той, которая поступила раньше. Для этих двух правил нужны разные операции над контейнером. Разберём их и напишем три небольшие функции.

## Цели

- узнавать в условии порядок LIFO или FIFO;
- использовать список как стек, а `collections.deque` — как очередь;
- выбирать поведение для пустого контейнера;
- применять эти конструкции к отмене действий, вложенным скобкам и обслуживанию заявок.

## Перед началом

Нужны функции, циклы, строки, списки, кортежи и исключения. Классы, декораторы и генераторы не используются. Все ячейки работают на Python 3.14 со стандартной библиотекой; внешние файлы не нужны.

Это отдельный практикум для семинаров. Он дополняет [основные занятия](../README.md) и может разбираться независимо от лекции об областях видимости и декораторах.

В заготовках стоит `pass`. Напишите функцию, выполните ячейку с проверками и уберите комментарий перед вызовом `check_task_N()`. Исходный `Run All` проходит, поскольку вызовы проверок закомментированы; готовых решений в ноутбуке нет.

## Два правила извлечения

Стек и очередь описывают **порядок доступа к элементам**. Под это правило выбирают контейнер и операции.

| Правило | Что извлекаем первым | Пример |
| --- | --- | --- |
| Стек, LIFO: last in, first out | Последнее добавленное | Отмена последнего изменения |
| Очередь, FIFO: first in, first out | Первое добавленное | Обслуживание заявок по порядку поступления |

Если добавить `A`, затем `B`, затем `C`, стек отдаст `C`, `B`, `A`, а очередь — `A`, `B`, `C`.

## Стек: работаем с правым концом списка

`append()` добавляет элемент в конец, `pop()` без аргумента снимает последний элемент и возвращает его. Посмотреть верхний элемент без удаления можно через `stack[-1]`.

В истории редактора сначала нужно отменить удаление строки: оно произошло последним. До запуска назовите, что останется в истории после `pop()`.

In [ ]:
history = ["добавлена строка", "изменён заголовок"]
history.append("удалена строка")

last_change = history.pop()
print("Отменяем:", last_change)
print("Осталось:", history)

assert last_change == "удалена строка"
assert history == ["добавлена строка", "изменён заголовок"]
assert history[-1] == "изменён заголовок"

## Очередь: добавляем справа, забираем слева

Для очереди используем `deque` из `collections`. `append()` добавляет в конец, `popleft()` извлекает первый элемент.

Заявка `r-1` должна обслужиться раньше `r-2`, хотя после неё уже успели добавить `r-3`. Куда попадёт новая заявка, если добавить её после первого обслуживания?

In [ ]:
from collections import deque

requests = deque(["r-1", "r-2"])
requests.append("r-3")

first = requests.popleft()
print("Обслужена:", first)
print("Остались:", list(requests))

assert first == "r-1"
assert list(requests) == ["r-2", "r-3"]

## Стоимость операций и пустой контейнер

| Действие | Стек через `list` | Очередь через `deque` |
| --- | --- | --- |
| Добавить | `append()`, амортизированная `O(1)` | `append()`, `O(1)` |
| Извлечь | `pop()`, амортизированная `O(1)` | `popleft()`, `O(1)` |
| Проверить пустоту | `not stack`, `O(1)` | `not queue`, `O(1)` |

Амортизированная оценка означает, что серия из `n` добавлений и извлечений обходится в `O(n)`, хотя отдельная операция со списком может потребовать переразмещения его внутреннего хранилища.

`list.pop(0)` тоже забирает первый элемент, но сдвигает оставшиеся элементы списка: одна операция занимает `O(n)`. Если последовательно опустошить так большой список, получится `O(n²)`. Операции `deque` на концах не требуют этого сдвига.

`[].pop()` и `deque().popleft()` поднимают `IndexError`. Поэтому договор функции должен определить, что значит попытка извлечения из пустого контейнера. Ниже пустая отмена и обслуживание пустой очереди ничего не делают: это явно заданное правило наших задач.

## Задание 1. История отмены

Напишите `apply_history(commands)`. Команды — пары `(action, value)`:

- `("add", value)` добавляет строку в историю;
- `("undo", None)` отменяет последнее неотменённое добавление;
- если история пуста, `undo` ничего не меняет;
- неизвестное `action` приводит к `ValueError`.

Верните список оставшихся строк в порядке их добавления. Пустой вход даёт новый пустой список. Повторы значений разрешены; отменяется отдельное добавление, а не все равные строки. Исходный список команд не изменяйте.

Пример: `[("add", "A"), ("add", "B"), ("undo", None), ("add", "C")]` даёт `["A", "C"]`.

Типы аргументов соответствуют аннотации; каждая команда содержит ровно два поля. Для `add` значение гарантированно строка, для `undo` — `None`. Не более 100 000 команд; длина каждой строки до 100 символов. Неизвестные действия могут встречаться и должны быть отклонены. Используйте список как стек. Требуется `O(n)` времени и `O(n)` памяти с учётом результата.

In [ ]:
def apply_history(commands: list[tuple[str, str | None]]) -> list[str]:
    pass

In [ ]:
def check_task_1():
    assert apply_history([]) == []
    assert apply_history([("undo", None), ("undo", None)]) == []
    commands = [("add", "A"), ("add", "B"), ("undo", None), ("add", "C")]
    assert apply_history(commands) == ["A", "C"]
    assert commands == [("add", "A"), ("add", "B"), ("undo", None), ("add", "C")]
    assert apply_history([("add", "A"), ("add", "A"), ("undo", None)]) == ["A"]
    assert apply_history([("add", ""), ("undo", None), ("undo", None)]) == []
    first = apply_history([])
    second = apply_history([])
    assert first is not second
    try:
        apply_history([("add", "A"), ("remove", None)])
    except ValueError:
        pass
    else:
        raise AssertionError("Неизвестное действие нужно отклонить")
    print("Задание 1: проверки пройдены")

# check_task_1()

## Задание 2. Вложенные скобки

Напишите `brackets_are_valid(text)`: верните `True`, если скобки `()`, `[]`, `{}` согласованы по типу и вложенности. Все остальные символы игнорируются.

Пустая строка корректна. Лишняя закрывающая, несовпадающая пара или оставшаяся открывающая скобка делают строку некорректной. Это проверка последовательности скобок, а не парсер Python: скобки внутри кавычек тоже учитываются.

Примеры: `"a * (b + c[0])"` и `"([{}])"` корректны, `"([)]"` и `"x)"` — нет.

Длина текста — до 100 000 символов. Используйте стек. Требуется `O(n)` времени и `O(n)` дополнительной памяти в худшем случае. Функция возвращает `bool`; несогласованные скобки — результат `False`, а не исключение.

In [ ]:
def brackets_are_valid(text: str) -> bool:
    pass

In [ ]:
def check_task_2():
    for text in ["", "только текст", "a * (b + c[0])", "([{}])", "()[]{}"]:
        assert brackets_are_valid(text) is True, text
    for text in ["([)]", "x)", "(", "(()", "][", "{[}]", '"("']:
        assert brackets_are_valid(text) is False, text
    assert brackets_are_valid("(" * 1000 + ")" * 1000) is True
    print("Задание 2: проверки пройдены")

# check_task_2()

## Задание 3. Обслуживаем заявки

Напишите `process_queue(events)`. События — пары `(action, request_id)`:

- `("arrive", request_id)` добавляет заявку в конец очереди;
- `("serve", None)` обслуживает первую заявку и добавляет её идентификатор в список обслуженных;
- обслуживание пустой очереди ничего не меняет;
- неизвестное действие приводит к `ValueError`.

Верните пару `(обслуженные, оставшиеся)`. Оба результата — новые списки идентификаторов: первый в порядке обслуживания, второй в порядке будущего обслуживания. Повторные идентификаторы означают разные заявки и не удаляются как дубликаты. Исходный список событий не изменяйте.

Пример: `[("arrive", "A"), ("arrive", "B"), ("serve", None), ("arrive", "C")]` даёт `(["A"], ["B", "C"])`.

Типы аргументов соответствуют аннотации, каждое событие содержит ровно два поля. Для `arrive` идентификатор гарантированно строка, для `serve` — `None`. Не более 100 000 событий; идентификаторы длиной до 100 символов. Используйте `deque`, вызов `list.pop(0)` запрещён. Требуется `O(n)` времени и `O(n)` памяти с учётом двух списков результата.

In [ ]:
def process_queue(
    events: list[tuple[str, str | None]],
) -> tuple[list[str], list[str]]:
    pass

In [ ]:
def check_task_3():
    assert process_queue([]) == ([], [])
    assert process_queue([("serve", None), ("serve", None)]) == ([], [])
    events = [("arrive", "A"), ("arrive", "B"), ("serve", None), ("arrive", "C")]
    assert process_queue(events) == (["A"], ["B", "C"])
    assert events == [("arrive", "A"), ("arrive", "B"), ("serve", None), ("arrive", "C")]
    assert process_queue([("arrive", "A"), ("arrive", "A"), ("serve", None)]) == (["A"], ["A"])
    assert process_queue([("arrive", ""), ("serve", None), ("serve", None)]) == ([""], [])
    served, remaining = process_queue([])
    assert served is not remaining
    try:
        process_queue([("arrive", "A"), ("skip", None)])
    except ValueError:
        pass
    else:
        raise AssertionError("Неизвестное действие нужно отклонить")
    print("Задание 3: проверки пройдены")

# check_task_3()

## Обсудим решения

1. Где в каждой функции видно правило LIFO или FIFO?
2. Почему множество не подходит для истории отмены и очереди заявок?
3. Что получится, если у очереди заменить `popleft()` на `pop()`?
4. В какой из трёх задач пустой контейнер — допустимое состояние, а в какой он помогает обнаружить ошибку во входе?
5. Почему правильные ответы на маленьких примерах ещё не доказывают линейную сложность решения?

## Итоги

Стек возвращает последнее добавленное, очередь — первое. В Python для стека достаточно правого конца списка, для очереди подходит `deque` с добавлением справа и извлечением слева.

Выбор операций определяет поведение и стоимость алгоритма. Поведение при пустоте задаётся условием: стандартный контейнер поднимает `IndexError`, а прикладная функция может заранее выбрать другой исход.

[Все справочные материалы](README.md) · [Занятие 9: области видимости и декораторы](../lesson09/README.md)

## Источники

- [Списки как стек и очередь — Python 3.14](https://docs.python.org/3.14/tutorial/datastructures.html#using-lists-as-stacks).
- [`collections.deque` — Python 3.14](https://docs.python.org/3.14/library/collections.html#collections.deque).